# MODELOS SIMPLES
## Regresión Logística Multinomial (Softmax)

En esta primera fase del proyecto, establecemos nuestra línea base (baseline) utilizando un modelo lineal. Aunque Fashion-MNIST es un dataset más complejo que el MNIST de dígitos original, el modelo de Regresión Softmax nos permitirá entender el límite inferior de rendimiento que podemos esperar.


## 1. Preparación de los datos


Como vimos en el EDA, las imágenes son de $28 \times 28$. Para un modelo lineal, debemos "aplanarlas" (flatten) a un vector de $784$ dimensiones.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras.datasets import fashion_mnist

# 1. Carga de datos (Siguiendo tus pasos del EDA)
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = fashion_mnist.load_data()

# 2. Preprocesamiento: Aplanado (Flatten) y Normalización
# Para modelos lineales, necesitamos vectores de 784 elementos (28x28)
X_all = x_train_raw.reshape(60000, 784) / 255.0
y_all = y_train_raw

X_test = x_test_raw.reshape(10000, 784) / 255.0
y_test = y_test_raw

# 3. DIVISIÓN DE DATOS
# Separamos el set de entrenamiento original en Train y Val (80/20)
X, X_val, y, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

print(f"X (Train): {X.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
X (Train): (48000, 784), X_val: (12000, 784), X_test: (10000, 784)


## 2. Definición del Modelo Softmax y Entrenamiento

En este modelo, buscamos minimizar la función de coste de Entropía Cruzada (Cross-Entropy):

$$J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \sum_{k=1}^{K} y_k^{(i)} \log(\hat{p}_k^{(i)})$$

Donde $\hat{p}_k^{(i)}$ es la probabilidad estimada de que la instancia $i$ pertenezca a la clase $k$, calculada mediante la función Softmax:

$$\hat{p}_k = \sigma(\mathbf{s}(\mathbf{x}))_k = \frac{e^{s_k(\mathbf{x})}}{\sum_{j=1}^{K} e^{s_j(\mathbf{x})}}$$

In [ ]:
# 4. Implementación del Modelo Lineal (Regresión Softmax)
# Usamos 'multinomial' para que sea un modelo Softmax real
model_linear = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=100)
model_linear.fit(X, y)



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(multi_class='multinomial')

## 3. Evaluación Final

In [ ]:
# 5. Evaluación de resultados
train_acc = accuracy_score(y, model_linear.predict(X))
val_acc = accuracy_score(y_val, model_linear.predict(X_val))
test_acc = accuracy_score(y_test, model_linear.predict(X_test))

# 6. CÁLCULO DE PARÁMETROS
# Un modelo lineal tiene: (n_características * n_clases) + n_biases
n_params = model_linear.coef_.size + model_linear.intercept_.size

print(f"\nResultados Modelo Lineal:")
print(f"Train Acc: {train_acc:.4f}")
print(f"Val Acc: {val_acc:.4f}")
print(f"Test Acc: {test_acc:.4f}")
print(f"Número de parámetros: {n_params}")


Resultados Modelo Lineal:
Train Acc: 0.8684
Val Acc: 0.8537
Test Acc: 0.8430
Número de parámetros: 7850


## Resumen de Resultados - Modelo Lineal

| Modelo | Parámetros |Train Acc | Val Acc | Test Acc|
| :--- | :---: | :---: | :---: |:--- |
| **Softmax (Baseline)** | 7850 | 0.8684 | 0.8537 |0.8430|0.8801|

### Análisis del Modelo Regresión Logística y Comparativa

El modelo ha aprendido bien los patrones generales del dataset.

La diferencia entre el entrenamiento y el test es de apenas un 2.5%. Esto indica que el modelo tiene un bajo sobreajuste (overfitting). Es un comportamiento esperado en modelos lineales, ya que su capacidad de memorización es limitada.

Hemos tenido bastante éxito al superar el 84% en Test con un modelo tan simple. Como vimos en el EDA, existen clases con alta correlación visual ('Shirt', 'T-shirt/top' y 'Pullover' > 0.90) que seguramente son las que están limitando que el accuracy sea mayor.
